# Medical AI — Fine-tuning LoRA (PoC)

Dans le cadre du projet TECHCORP, une mission expérimentale consiste à développer un modèle conversationnel médical à partir d’un modèle de base.
L’objectif est de spécialiser un modèle généraliste afin d’améliorer la pertinence des réponses sur des questions médicales simples.

## Approche

Le fine-tuning est réalisé à l’aide de la méthode **LoRA (Low-Rank Adaptation)**, qui permet d’adapter un modèle de langage de manière efficace sans réentraîner l’ensemble des paramètres.

Le dataset utilisé provient de conversations médicales entre patients et professionnels de santé. Source : https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot

In [1]:
# On vérifie que le GPU est bien disponible et affiche ses caractéristiques (mémoire, modèle)
!nvidia-smi

Fri Apr 24 11:55:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Installation des dépendances

In [2]:
# Installe toutes les librairies nécessaires au fine-tuning LoRA
!pip install -q datasets transformers accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.8 MB/s eta 0:00:00


## Exploration du dataset médical
Le dataset utilisé contient des conversations médicales entre patients et médecins. Il est chargé depuis hugging face et analysé pour comprendre sa structure.

In [3]:
# Téléchargement et chargement du dataset médical depuis HuggingFace
from datasets import load_dataset
dataset = load_dataset("ruslanmv/ai-medical-chatbot")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/863 [00:00<?, ?B/s]

dialogues.parquet:   0%|          | 0.00/142M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/256916 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Description', 'Patient', 'Doctor'],
        num_rows: 256916
    })
})

In [4]:
# Visualisation d'exemples pour comprendre la sctucture du dataset
dataset["train"][0]

{'Description': 'Q. What does abutment of the nerve root mean?',
 'Patient': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?',
 'Doctor': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}

## Préparation du dataset
On prépare le dataset en vue de son utilisation pour le fine tuning

In [5]:
# On renomme les colonnes "Patient" et "Doctor" du dataset en "instruction" et "response"
def format_example(example):
    return {
        "instruction": example["Patient"],
        "response": example["Doctor"]
    }

formatted = dataset["train"].map(format_example)
formatted[0]

Map:   0%|          | 0/256916 [00:00<?, ? examples/s]

{'Description': 'Q. What does abutment of the nerve root mean?',
 'Patient': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?',
 'Doctor': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->',
 'instruction': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?',
 'response': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}

In [6]:
# On supprime les espaces inutiles et retire les entrées vides du dataset
def clean_example(example):
    instruction = example["instruction"].strip()
    response = example["response"].strip()

    if instruction == "" or response == "":
        return None

    return {
        "instruction": instruction,
        "response": response
    }

cleaned = formatted.map(clean_example)
cleaned = cleaned.filter(lambda x: x is not None)
cleaned[0]

Map:   0%|          | 0/256916 [00:00<?, ? examples/s]

Filter:   0%|          | 0/256916 [00:00<?, ? examples/s]

{'Description': 'Q. What does abutment of the nerve root mean?',
 'Patient': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?',
 'Doctor': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->',
 'instruction': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?',
 'response': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}

In [7]:
# Garde uniquement les colonnes utiles et supprime les colonnes d'origine devenues inutiles
cleaned = cleaned.map(lambda x: {
    "instruction": x["instruction"],
    "response": x["response"]
})

cleaned = cleaned.remove_columns(["Description", "Patient", "Doctor"])

cleaned[0]

Map:   0%|          | 0/256916 [00:00<?, ? examples/s]

{'instruction': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?',
 'response': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}

In [8]:
# Ajout d’un message de précaution dans chaque réponse.
def add_safety(example):
    return {
        "instruction": example["instruction"],
        "response": example["response"] + " Please consult a healthcare professional for medical advice."
    }

final_dataset = cleaned.map(add_safety)

final_dataset[0]

Map:   0%|          | 0/256916 [00:00<?, ? examples/s]

{'instruction': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?',
 'response': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online --> Please consult a healthcare professional for medical advice.'}

In [9]:
# Sauvegarde. Export du dataset final au format JSONL pour le fine-tuning.
final_dataset.to_json("medical_dataset_clean.jsonl")

Creating json from Arrow format:   0%|          | 0/257 [00:00<?, ?ba/s]

274928041

## Préparation pour le fine tuning

In [10]:
# On réduit le dataset pour réaliser un fine-tuning rapide dans le cadre du POC.
# Sélectionne aléatoirement 2000 exemples du dataset
small_dataset = final_dataset.shuffle(seed=42).select(range(2000))
small_dataset

Dataset({
    features: ['instruction', 'response'],
    num_rows: 2000
})

In [11]:
# Formate chaque exemple en un texte structuré "Instruction / Response" pour l'entraînement
def format_prompt(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"
    }

train_dataset = small_dataset.map(format_prompt)
train_dataset[0]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'instruction': 'last year my wife was went through a surgery for appendix cancer, that appendix was removed , that appendix slice tested in lab and found so called adino carcinoma in apedix,  after that doctor decided to operate again and remove her partial intestine, there was no sign of cancer in any test other than the biopsy of appendix, however  after one moth of hospitalization came back to home, 6 month of follow up check up no bad sign, now almost one year of surgery puss mark notice at steches near belly button .Please advice this is not a sign of any cancer',
 'response': 'Hi and welcome to HCM. First, you dont have to worry. This cant be tumour relaps because this is lesion in abdominall wall,obviously some local infection or wound abscess.This is often seen after laparotomy. Appendix cancers are rare but in most cases surgery is enough for complete treatment and recovery. If tehre were no found metastasis or extended disease during the surgery, you dont have to expect rela

## Chargement du modèle de base
Chargement du modèle Phi-3.5 pour le fine-tuning LoRA.

In [13]:
# Charge le modèle Phi-3 en 4-bit avec une quantization NF4 optimisée pour le fine-tuning
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [14]:
# Configuration de l’adapter LoRA pour entraîner uniquement une petite partie du modèle.
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["qkv_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

lora_config

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'o_proj', 'qkv_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)

In [16]:
# Ajout de l’adapter LoRA au modèle de base.

from peft import get_peft_model

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 4,718,592 || all params: 3,825,798,144 || trainable%: 0.1233


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'microsoft/phi-3-mini-4k-instruct' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


## Configuration de l’entraînement

In [20]:
# Définition des paramètres d’entraînement pour un fine-tuning
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./medical_lora_model",  # dossier de sauvegarde
    per_device_train_batch_size=1,      # petit batch pour GPU T4
    gradient_accumulation_steps=4,      # simule un batch plus grand
    learning_rate=2e-4,                 # taux d’apprentissage LoRA
    num_train_epochs=1,                 # 1 epoch pour le PoC
    logging_steps=10,                   # logs réguliers
    save_steps=100,                     # sauvegarde régulière
    fp16=False,                         # désactivé pour éviter l’erreur AMP
    bf16=False,                         # désactivé car T4 ne gère pas bien bf16
    optim="paged_adamw_8bit",           # optimiseur mémoire légère
    report_to="none"                    # pas de wandb
)

training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,

In [21]:
# Initialisation du Trainer. Prend le modèle, les données et les paramètres, puis gère automatiquement le fine-tuning
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,                 # modèle Phi-3 + LoRA
    args=training_args,          # paramètres d'entraînement définis avant
    train_dataset=train_dataset, # dataset formaté avec "text"
    processing_class=tokenizer,  # tokenizer → transforme texte en tokens pour le modèle
)

## Entraînement du modèle

Le fine-tuning LoRA est réalisé sur un sous-ensemble du dataset médical afin de produire un Proof of Concept.

L'entraînement consiste à ajuster uniquement une petite partie des paramètres du modèle (≈0.1%), ce qui permet de réduire fortement le temps et les sources nécessaires.

La métrique principale suivie est la loss, qui mesure l'écart entre les réponses générées et les réponses attendues.

In [22]:
# Lancement du fine-tuning LoRA sur le dataset médical (PoC).
trainer.train()

Step,Training Loss
10,2.648638
20,2.418881
30,2.361242
40,2.264814
50,2.270586
60,2.317891
70,2.287049
80,2.142420
90,2.221354
100,2.268248


TrainOutput(global_step=500, training_loss=2.213716796875, metrics={'train_runtime': 1421.046, 'train_samples_per_second': 1.407, 'train_steps_per_second': 0.352, 'total_flos': 1.2383789516863488e+16, 'train_loss': 2.213716796875})

In [24]:
# Sauvegarde du modèle
trainer.save_model("./medical_lora_model")
tokenizer.save_pretrained("./medical_lora_model")

('./medical_lora_model/tokenizer_config.json',
 './medical_lora_model/chat_template.jinja',
 './medical_lora_model/tokenizer.json')

## Test du modèle
Test du modèle sur quelques questions médicales simples afin d’évaluer son comportement conversationnel.

In [27]:
# Fonction de test
def generate_answer(question):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [28]:
# Question de test 1
print(generate_answer("What are common symptoms of flu?"))

### Instruction:
What are common symptoms of flu?

### Response:
Hi,Flu or influenza is a viral infection that affects the respiratory tract. It is caused by influenza A and B viruses. The symptoms of flu are:Fever, chills, cough, sore throat, runny or stuffy nose, muscle and body aches, headache, fatigue, and sometimes vomiting and diarrhea. The symptoms usually last for 1 to 2 weeks. The flu can be prevented by getting vaccinated. Please consult a healthcare professional for medical advice.


In [29]:
# Question de test 2
print(generate_answer("What is diabetes?"))

### Instruction:
What is diabetes?

### Response:
Hi. Diabetes is a condition in which the body is unable to produce enough insulin or use it properly. Insulin is a hormone that helps the body use sugar for energy. When the body does not have enough insulin or cannot use it properly, sugar builds up in the blood. This can cause many health problems. There are two main types of diabetes. Type 1 diabetes is when the body does not produce enough insulin. Type 2 diabetes is when the body does not use insulin properly. Both types of diabetes can be controlled with medication and lifestyle changes. If you have any more questions, feel free to ask. Thanks.


In [30]:
# Question de test 3
print(generate_answer("Can you diagnose chest pain?"))

### Instruction:
Can you diagnose chest pain?

### Response:
Hi,Thanks for using health care magic.I can help you in diagnosing the chest pain.I need to know more about the chest pain.Is it sharp or dull?Is it constant or intermittent?Is it associated with any other symptoms like shortness of breath, palpitation, sweating, nausea or vomiting?Is it associated with any activity like exercise, walking or lying down?Is it associated with any other symptoms like cough, fever, sore throat or wheezing?Is it associated with any other symptoms like heartburn, indigestion or acid reflux?Is it associated with any other symptoms like anxiety or stress?Is it


In [33]:
# Champ libre pour poser vos questions
question = input("Enter your medical question: ")

answer = generate_answer(question)

print("\n--- Model Answer ---\n")
print(answer)

Enter your medical question: My nose is red

--- Model Answer ---

Hi,You have a redness in your nose. It may be due to allergy. You can apply petroleum jelly on the affected area. If it persists, then consult a doctor. Hope I have answered your query. Let me know if I can assist you further. Regards,Dr. Sumanth M. Please consult a healthcare professional for medical advice.
